# AraBERT Noun Attachment: Training, Validation & Evaluation (Human Gold Corpus)

## 1. Install Dependencies & Setup Environment

In [ ]:
!pip install -q transformers datasets arabert farasa pyarabic scikit-learn pandas openpyxl torch

In [ ]:
import io
import torch
import numpy as np
import pandas as pd
from google.colab import files
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from arabert.preprocess import ArabertPreprocessor

# Set seed for exact reproducibility
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Upload Human-Annotated Datasets

In [ ]:
print("📌 UPLOAD 1/3: Select 'training_dataset.xlsx' (Human Labeled)")
up_train = files.upload()
df_train = pd.read_excel(io.BytesIO(up_train[list(up_train.keys())[0]]))

print("\n📌 UPLOAD 2/3: Select 'valid_dataset.xlsx' (Human Labeled)")
up_val = files.upload()
df_val = pd.read_excel(io.BytesIO(up_val[list(up_val.keys())[0]]))

print("\n📌 UPLOAD 3/3: Select 'eval_dataset.xlsx' (Human Labeled)")
up_eval = files.upload()
df_eval = pd.read_excel(io.BytesIO(up_eval[list(up_eval.keys())[0]]))

# Map target string labels (N1 -> 0, N2 -> 1) for each dataframe
label_map = {"N1": 0, "N2": 1}
for df in [df_train, df_val, df_eval]:
    label_col = "human_gold_attachment" if "human_gold_attachment" in df.columns else "attachment"
    df["label"] = df[label_col].astype(str).str.upper().str.strip().map(label_map)

print(f"\n✓ Train Set: {len(df_train):,} rows | Valid Set: {len(df_val):,} rows | Eval Set: {len(df_eval):,} rows\n")

## 3. Preprocessing & Tokenization

In [ ]:
print("📌 Preprocessing Arabic text with Farasa & AraBERT Tokenizer...")
arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def format_input(row):
    sent = arabert_prep.preprocess(str(row["sentence"]))
    n1 = arabert_prep.preprocess(str(row["head_N1"]))
    n2 = arabert_prep.preprocess(str(row["complement_N2"]))
    return f"{sent} [SEP] N1: {n1} [SEP] N2: {n2}"

for df in [df_train, df_val, df_eval]:
    df["input_text"] = df.apply(format_input, axis=1)

class AttachmentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_ds = AttachmentDataset(df_train["input_text"].tolist(), df_train["label"].tolist())
val_ds = AttachmentDataset(df_val["input_text"].tolist(), df_val["label"].tolist())
eval_ds = AttachmentDataset(df_eval["input_text"].tolist(), df_eval["label"].tolist())

## 4. Setup Model, Metrics & Training Parameters

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1_binary, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_binary": f1_binary,
        "f1_weighted": f1_weighted
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="./arabert_pure_human_gold",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

## 5. Fine-Tuning & Model Training

In [ ]:
print("🚀 Fine-tuning AraBERT on human-annotated training partition...")
trainer.train()

## 6. Model Evaluation & Detailed Classification Report

In [ ]:
print("\n📌 Running evaluation on unseen human-annotated eval_dataset.xlsx...")
eval_preds = trainer.predict(eval_ds)
results = eval_preds.metrics

print("\n==================================================")
print("     FINAL METRICS (PURE HUMAN GROUND TRUTH)     ")
print("==================================================")
print(f" ACCURACY:     {results.get('test_accuracy', results.get('eval_accuracy', 0)) * 100:.2f}%")
print(f" PRECISION:    {results.get('test_precision', results.get('eval_precision', 0)) * 100:.2f}%")
print(f" RECALL:       {results.get('test_recall', results.get('eval_recall', 0)) * 100:.2f}%")
print(f" BINARY F1:    {results.get('test_f1_binary', results.get('eval_f1_binary', 0)) * 100:.2f}%")
print(f" WEIGHTED F1:  {results.get('test_f1_weighted', results.get('eval_f1_weighted', 0)) * 100:.2f}%")
print("==================================================")

pred_labels = np.argmax(eval_preds.predictions, axis=1)

print("\n--- Detailed Classification Report ---")
print(classification_report(df_eval["label"], pred_labels, target_names=["N1", "N2"], digits=4))

cm = confusion_matrix(df_eval["label"], pred_labels)
print("--- Confusion Matrix ---")
print(f"True N1 -> Predicted N1: {cm[0][0]:>6} | Predicted N2: {cm[0][1]:>6}")
print(f"True N2 -> Predicted N1: {cm[1][0]:>6} | Predicted N2: {cm[1][1]:>6}")